In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
train=pd.read_csv('/kaggle/input/is54a-tri-tu-nhan-to-bav-itde-2026/train.csv')
test=pd.read_csv('/kaggle/input/is54a-tri-tu-nhan-to-bav-itde-2026/test.csv')
print(train.shape,test.shape)

(6000, 54) (4000, 53)


In [ ]:
X = train.drop(columns=['Academic_Status'])
y = train['Academic_Status']

# bỏ ID khi train
X = X.drop(columns=['Student_ID'])
test_ids = test['Student_ID']
X_test = test.drop(columns=['Student_ID'])

print(X.shape)
print(y.shape)

(6000, 52)
(6000,)


In [ ]:
ca_cols=X.select_dtypes(include=['object']).columns.tolist()
num_cols=X.select_dtypes(exclude=['object']).columns.tolist()
print('Danh muc:',len(ca_cols))
print('So:',len(num_cols))


Danh muc: 8
So: 44


In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ca_cols),
        ('num', 'passthrough', num_cols)
    ]
)

model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42
)

pipeline = Pipeline(
    steps=[
        ('preprocess', preprocess),
        ('model', model)
    ]
)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y)

pipeline.fit(X_train, y_train)

val_pred = pipeline.predict(X_val)
print("Validation accuracy:", accuracy_score(y_val, val_pred))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002713 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1269
[LightGBM] [Info] Number of data points in the train set: 4800, number of used features: 206
[LightGBM] [Info] Start training from score -0.441740
[LightGBM] [Info] Start training from score -1.499090
[LightGBM] [Info] Start training from score -2.011783
Validation accuracy: 0.8566666666666667


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
test_pred = pipeline.predict(X_test)

submission = pd.DataFrame({
    'Student_ID': test_ids,
    'Academic_Status': test_pred
})

submission.to_csv('submission.csv', index=False)
submission.head()


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,Student_ID,Academic_Status
0,SV20211602,1
1,SV20215385,0
2,SV20217528,1
3,SV20218254,1
4,SV20210560,0
